# The pipeline on one synthetic complaint

raw text → normalization → BPE → word2vec → seq2seq (Task B). The complaint below is written for this demo; it is not real.

In [1]:
# setup: run from the repository root (python notebooks/build.py)
import sys; sys.path.insert(0, '.')

In [2]:
raw = ('On XX/XX/2021 a debt collector called me about {$1,250.00} I do not owe. '
       'I sent a validation letter under the FDCPA to XXXX XXXX but they kept calling my work.')
from corpus.normalize import normalize
text = normalize(raw)
print(text)

On <DATE> a debt collector called me about <AMT> I do not owe. I sent a validation letter under the FDCPA to <REDACTED> but they kept calling my work.


In [3]:
from bpe.tokenizer import Tokenizer
tok = Tokenizer.load('bpe/artifacts/cfpb_8k')
ids = tok.encode(text)
print(len(ids), 'tokens:', tok.pieces(text))

34 tokens: ['On', '▁', '<DATE>', '▁a', '▁debt', '▁collector', '▁called', '▁me', '▁about', '▁', '<AMT>', '▁I', '▁do', '▁not', '▁owe', '.', '▁I', '▁sent', '▁a', '▁validation', '▁letter', '▁under', '▁the', '▁FDCPA', '▁to', '▁', '<REDACTED>', '▁but', '▁they', '▁kept', '▁calling', '▁my', '▁work', '.']


In [4]:
from word2vec.model import Embeddings
from word2vec.vocab import tokenize
emb = Embeddings.load('word2vec/artifacts/seed0')
words = [w for w in tokenize(text) if w in emb.index]
print('in-vocabulary words:', len(words), 'of', len(tokenize(text)))
print('neighbors of "collector":', [n for n, _ in emb.most_similar('collector', 5)])

in-vocabulary words: 29 of 29
neighbors of "collector": ['debt', 'sumer', 'thirty-day', 'collectors', 'col']


In [5]:
import json, torch
from seq2seq.data_taskB import TaskB, truncate_word_safe
from seq2seq.model import Seq2Seq
from seq2seq.decode import Trie, beam_search
from seq2seq.train import pad_batch
run = json.loads(open('seq2seq/runs/taskB_seed0/run.json').read())
task = TaskB(run['cfg']['tokenizer'], run['cfg']['L'])
hp = run['hparams']
model = Seq2Seq(hp['src_vocab'], hp['tgt_vocab'], emb=hp['emb'], hid=hp['hid'], att=hp['att'], attention=hp['attention'])
model.load_state_dict(torch.load('seq2seq/runs/taskB_seed0/model.pt')); model.eval()
src = truncate_word_safe(task.tok.encode(text), run['cfg']['L'], task.tok)
x, xl = pad_batch([src])
out = beam_search(model, x, xl, width=5, trie=Trie(task.label_ids.values()))
print('predicted label:', task.to_label(out))

predicted label: Debt collection | Communication tactics
